In [2]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()
import requests


## Langchain Integration
https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/gen_ai_hub.html#langchain-integration

### Harmonized Model Initialization
The init_llm and init_embedding_model functions allow easy initialization of langchain model interfaces in a harmonized way in generative AI hub sdk

In [3]:
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


template = """Question: {question}
    Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=['question'])
question = 'What is a supernova?'

chain = prompt | model | StrOutputParser()
response = chain.invoke({'question': question})
print(response)

A supernova is a powerful and luminous explosion that occurs during the last evolutionary stages of a massive star's life cycle or when a white dwarf is triggered into runaway nuclear fusion. Let's break it down step by step:

1. **Types of Supernovae**: There are two primary types of supernovae:
   - **Type I Supernova**: This occurs in binary star systems. A white dwarf star accumulates matter from its companion star. When the white dwarf reaches a critical mass (the Chandrasekhar limit), it undergoes a runaway nuclear reaction, leading to an explosion.
   - **Type II Supernova**: This happens when a massive star (at least eight times the mass of the Sun) exhausts its nuclear fuel. The core collapses under gravity, and the outer layers are expelled in a massive explosion.

2. **Core Collapse**: In massive stars, once nuclear fusion in the core ceases, the core collapses under its own gravity. This collapse can lead to the formation of a neutron star or black hole, depending on the ma

init_embedding_model

In [107]:
from gen_ai_hub.proxy.langchain.init_models import init_embedding_model

text = 'Every decoding is another encoding.'

embeddings = init_embedding_model('text-embedding-3-large')
response = embeddings.embed_query(text)
#print(response)


### Structured model outputs

In [109]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.prompts.chat import HumanMessage
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
chat_model = ChatOpenAI(proxy_model_name="gpt-4o", proxy_client=get_proxy_client())
chat_model = chat_model.with_structured_output(method="json_schema", schema=Person, strict=True)

message = HumanMessage(content="Tell me about a person named John who is 30")
print(chat_model.invoke([message]))


name='John' age=30


## Agent

### Tool

#### Define tools

In [ ]:
from langchain.tools import tool

@tool
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]

#### Involke tools

When using a model separately from an agent, it is up to you to execute the requested tool and return the result back to the model for use in subsequent reasoning. 

In [59]:
model_with_tools = model.bind_tools([get_weather])  

response = model_with_tools.invoke("What's the weather like in Beijing and Shanghai?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'latitude': 39.9042, 'longitude': 116.4074}
Tool: get_weather
Args: {'latitude': 31.2304, 'longitude': 121.4737}


#### Tool execution loop

When a model returns tool calls, you need to execute the tools and pass the results back to the model. This creates a conversation loop where the model can use tool results to generate its final response.

In [ ]:
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [
    {
        "role": "user", 
        "content": "What's the weather in Boston and Shanghai?"
    }
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
print(f"Step 1:{ai_msg.tool_calls}")



# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    print(f"Step 2 tool_call:{tool_call}")
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    print(f"Step 2 messages:{messages}")


# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(f"Step 3:{final_response.text}")


Step 1:[{'name': 'get_weather', 'args': {'latitude': 42.3601, 'longitude': -71.0589}, 'id': 'call_1mLnFmRN6QVEe2K0LKdV1pSO', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'latitude': 31.2304, 'longitude': 121.4737}, 'id': 'call_MDbPzd2qFUB0PupPig8JfzkZ', 'type': 'tool_call'}]
Step 2 tool_call:{'name': 'get_weather', 'args': {'latitude': 42.3601, 'longitude': -71.0589}, 'id': 'call_1mLnFmRN6QVEe2K0LKdV1pSO', 'type': 'tool_call'}
Step 2 messages:[{'role': 'user', 'content': "What's the weather in Boston and Shanghai?"}, AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_1mLnFmRN6QVEe2K0LKdV1pSO', 'function': {'arguments': '{"latitude": 42.3601, "longitude": -71.0589}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 'call_MDbPzd2qFUB0PupPig8JfzkZ', 'function': {'arguments': '{"latitude": 31.2304, "longitude": 121.4737}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens

In [ ]:
content="additional_kwargs={
    'tool_calls': [
        {
            'id': 'call_7XrsS3NrnffMIkacVzUlQgoQ',
             'function': {
                'arguments': '{"latitude":31.2304,"longitude":121.4737,"city":"Shanghai"}', 
                'name': 'get_weather'
            }, 
            'type': 'function'
        }
    ], 
    'refusal': None
} 
response_metadata={
    'token_usage': {
        'completion_tokens': 29, 
        'prompt_tokens': 68, 
        'total_tokens': 97,
         'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-Cvdn6O7ql1pZjjn3pPnNg9o72iJYq', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019b9c4a-85b0-7023-a616-8d8f4f88f7b9-0' tool_calls=[{'name': 'get_weather', 'args': {'latitude': 31.2304, 'longitude': 121.4737, 'city': 'Shanghai'}, 'id': 'call_7XrsS3NrnffMIkacVzUlQgoQ', 'type': 'tool_call'}] usage_metadata={'input_tokens': 68, 'output_tokens': 29, 'total_tokens': 97, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

### Basic structure

#### Define tools

In [5]:
from langchain.tools import tool
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

#### Define agents

In [6]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)


agent = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Invoke agents

In [112]:

response =agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "What is weather in Shanghai?"
            }
        ]
    }
)
print(response)

 

{'messages': [HumanMessage(content='What is weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='6e5d6977-a882-46c5-a087-8c967baea53f'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QRr0sHsBOMT2e6xgcwK3yhnE', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 78, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CoPVkpuOC7UGlTCamEpRDHkOBzSpm', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b3594-c206-73f3-91b6-0ea537a849b1-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shanghai'}, 'id': 'c

##### Optional:study on the response structure

In [113]:
print(type(response))
len(response)
response['messages']

<class 'dict'>


[HumanMessage(content='What is weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='6e5d6977-a882-46c5-a087-8c967baea53f'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QRr0sHsBOMT2e6xgcwK3yhnE', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 78, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CoPVkpuOC7UGlTCamEpRDHkOBzSpm', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b3594-c206-73f3-91b6-0ea537a849b1-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shanghai'}, 'id': 'call_QRr0sHsB

In [114]:
messages=response['messages']
print(type(messages))
HumanMessage=messages[0]
AIMessage=messages[1]
ToolMessage=messages[2]
AIMessage_output=messages[-1]
AIMessage_output.content

<class 'list'>


'The weather in Shanghai is currently sunny with a temperature of 72°F.'

In [115]:
print(type(AIMessage))
print(json.dumps(AIMessage.additional_kwargs,indent=2))

<class 'langchain_core.messages.ai.AIMessage'>
{
  "tool_calls": [
    {
      "id": "call_QRr0sHsBOMT2e6xgcwK3yhnE",
      "function": {
        "arguments": "{\"location\":\"Shanghai\"}",
        "name": "get_weather"
      },
      "type": "function"
    }
  ],
  "refusal": null
}


In [116]:
tool_calls=AIMessage.additional_kwargs['tool_calls'][0]
print(type(tool_calls))
function=tool_calls['function']
function['name']

<class 'dict'>


'get_weather'

#### Check the message

Define a function to return message directly.

In [16]:

def invoke_agent_messages(agent, content: str):
    payload = {
        "messages": [
            {
                "role": "user", 
                "content": content
            }
        ]
    }
    response = agent.invoke(payload)
    return response["messages"]  # 若缺失会直接抛 KeyError


To check the contect in an easier way, we create a function to find out and then print out key information based on the structure of this message.

In [13]:
import json

def print_message_pairs(messages, verbose=False):
    """
    自动从消息序列中提取：
      - user_query：第一个 HumanMessage 的 content
      - tool_name：第一个 AIMessage.additional_kwargs.tool_calls[0].function.name
      - tool_output：第一个 ToolMessage 的 content
      - assistant_text：最后一个 AIMessage 的 content

    参数：
      - messages: 消息序列（包含 HumanMessage / AIMessage / ToolMessage 等）
      - verbose (bool): 
          True  -> 打印 JSON（包含四个键值）
          False -> 仅打印 assistant_text

    返回：
      - pairs (dict): 以上四个字段的字典，便于后续使用
    """
    # 安全提取工具名
    def extract_tool_name_from_ai(ai_msg):
        ak = getattr(ai_msg, "additional_kwargs", {})
        if isinstance(ak, dict):
            tool_calls = ak.get("tool_calls") or []
            if tool_calls:
                fn = tool_calls[0].get("function") or {}
                return fn.get("name")
        return None

    # 初始化
    user_query = None
    tool_name = None
    tool_output = None
    assistant_text = None

    # 1) 找第一个 HumanMessage 作为用户文本
    for m in messages:
        if m.__class__.__name__ == "HumanMessage":
            user_query = getattr(m, "content", None)
            break

    # 2) 找第一个 AIMessage 中的 tool_calls 取函数名
    for m in messages:
        if m.__class__.__name__ == "AIMessage":
            tool_name = extract_tool_name_from_ai(m)
            if tool_name:
                break

    # 3) 找第一个 ToolMessage 的输出
    for m in messages:
        if m.__class__.__name__ == "ToolMessage":
            tool_output = getattr(m, "content", None)
            break

    # 4) 找最后一个 AIMessage 的最终回复
    for m in reversed(messages):
        if m.__class__.__name__ == "AIMessage":
            assistant_text = getattr(m, "content", None)
            break

    pairs = {
        "user_query": user_query,
        "tool_name": tool_name,
        "tool_output": tool_output,
        #"assistant_text": assistant_text,
    }

    # 根据 verbose 控制打印
    if verbose:
        # 打印完整 JSON；ensure_ascii=False 支持中文直出（可按需移除）
        print(json.dumps(pairs, indent=2, ensure_ascii=False))
        print("\nAassistant reply:")
        print(assistant_text or "")
    else:
        # 仅打印最终助手回复；为防 None，做一下空串兜底
        print(assistant_text or "")




Now it is easier for us to see that, in the below case the tool [search] is not applied.

In [119]:
messages = invoke_agent_messages(agent, "Where is the location of Shanghai?")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Where is the location of Shanghai?",
  "tool_name": null,
  "tool_output": null
}

Aassistant reply:
Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea.


To let agent to use the tool, change the question closer to the tool description.

In [120]:
messages = invoke_agent_messages(agent, "Search for the location of Shanghai")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Search for the location of Shanghai",
  "tool_name": "search",
  "tool_output": "Results for: location of Shanghai"
}

Aassistant reply:
Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea.


### Dynamic system prompt

For more advanced use cases where you need to modify the system prompt based on runtime context or agent state, you can use middleware.<br>
The <i>@dynamic_prompt</i> decorator creates middleware that generates system prompts based on the model request:

In [136]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent_dyn = create_agent(
    model,
    tools = [search],
    middleware=[user_role_prompt],
    context_schema=Context
)


The system prompt will be set dynamically based on context so we get different answers to the same question.

In [138]:
query="Search for the explaination of context_schemaand and mmiddleware in Langchain Agent, and then interpret them further."

In [139]:
# When set user_role as "beginner"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content":query}]},
    context={"user_role": "beginner"}
)
messages=response["messages"]
print_message_pairs(messages)

I wasn't able to retrieve specific information about "context_schema" and "middleware" in Langchain Agent. However, I can provide a general explanation based on typical usage in programming and AI frameworks:

1. **Context Schema**:
   - In many frameworks, a context schema refers to the structure or format of the data that is passed around within the system. It defines what kind of information is included, how it is organized, and how it can be accessed or modified. In the context of Langchain or similar AI frameworks, a context schema might specify the types of inputs and outputs that an agent can handle, including any metadata or additional parameters that are necessary for processing.

2. **Middleware**:
   - Middleware generally refers to software that acts as a bridge between different systems or layers within an application. It can be used to manage data flow, handle requests, or perform operations like logging, authentication, and error handling. In the context of Langchain Age

In [140]:
# When set user_role as "expert"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": query}]},
    context={"user_role": "expert"}
)
messages=response["messages"]
print_message_pairs(messages)

It seems there was an issue retrieving the specific explanations for "context_schema" and "middleware" in Langchain Agent. However, I can provide a general interpretation based on typical usage in similar contexts:

### Context Schema in Langchain Agent

**Context Schema** typically refers to a structured format or blueprint that defines the context in which an agent operates. In the context of Langchain or similar frameworks, a context schema might include:

- **Variables and Parameters**: Definitions of the variables that the agent can use or modify during its operation.
- **Data Types**: Specifications of the types of data (e.g., strings, integers, objects) that the agent can handle.
- **Constraints**: Rules or conditions that the data must satisfy.
- **Relationships**: How different pieces of data relate to each other within the context.

In Langchain, a context schema would help in structuring the input and output data for agents, ensuring that they operate within defined paramete

### MiddleWare

#### Setup: model + tools

In [3]:
from langchain_core.tools import tool
# --- Define tools ---
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"



tools = [search, get_weather]

# --- Define model (replace your API key/config as needed) ---
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=4800
)

#### Define custom Context + middleware

In [4]:

from typing import TypedDict, Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware

class Context(TypedDict):
    user_preferences: dict  # {"style": "...", "verbosity": "..."}

class CustomMiddleware(AgentMiddleware):
    # (Optional) stage-specific tool restrictions:
    # tools = [tool1, tool2]

    def before_model(self, state, runtime) -> dict[str, Any] | None:
        # Read preferences from runtime context
        prefs = runtime.context.get("user_preferences", {}) or {}
        style = str(prefs.get("style", "general")).lower()
        verbosity = str(prefs.get("verbosity", "normal")).lower()

        # Base prompt
        system_prompt = "You are a helpful assistant."

        # Style-specific guidance
        if style == "technical":
            system_prompt += " Prefer precise, technical language and include implementation details."
        elif style == "casual":
            system_prompt += " Keep explanations informal, approachable, and friendly."

        # Verbosity-specific guidance
        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with concrete examples."
        elif verbosity in ("brief", "low"):
            system_prompt += " Be concise and focus on key points; use short sentences and bullet points where helpful."

        # Tune generation params (optional)
        temperature = 0.2 if style == "technical" else 0.7  # more deterministic for technical, more open for casual

        # Return updates for the upcoming model call
        return {
            "messages": [{"role": "system", "content": system_prompt}],
            "model_kwargs": {"temperature": temperature},
        }


#### Create agent

In [5]:

agent = create_agent(
    model,
    tools=tools,                       # e.g., [search, get_weather]
    middleware=[CustomMiddleware()],
    context_schema=Context,            # <-- use context, not state
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)


#### Invoke the agent with user_preferences

In [6]:
query="Search for the explaination vector embeddings." 
query="Search for the story lines and theme in 三国演义 in Chinese" 
query="Search for the story lines and theme in Games of Throne and then introduce these in Chinese." 

In [9]:
# A user who prefers technical & detailed responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "technical",
            "verbosity": "detailed"
        }
    }
)
messages = result["messages"]

print("\n=============================== Assistant reply (technical + detailed) =================================")
print_message_pairs(messages,verbose=True)



=============================== Assistant reply (technical + detailed) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

**Storylines:**
1. **The Iron Throne:** The central storyline revolves around the struggle for power and control of the Iron Throne of the Seven Kingdoms. Various noble families, including the Starks, Lannisters, Baratheons, and Targaryens, vie for dominance.
   
2. **The Stark Family:** The Starks of Winterfell face numerous challenges, including betrayal, political intrigue, and the fight to reclaim their home and honor.

3. **Daenerys Targaryen's Quest:** Daenerys Targaryen's journey from exile to power, as she seeks to reclaim the throne for her family, is marked by her growth as a leader and her acquisition 

In [130]:

# A user who prefers casual & brief responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "casual",
            "verbosity": "brief"
        }
    }
)
messages = result["messages"]


print("\n=============================== Assistant reply (casual + brief) =================================")
print_message_pairs(messages,verbose=True)




=============================== Assistant reply (casual + brief) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

- **Storylines:**
  - **Power Struggles:** The series revolves around the battle for the Iron Throne among noble families.
  - **Family Dynamics:** Focuses on the relationships and conflicts within families like the Starks, Lannisters, and Targaryens.
  - **Mystical Elements:** Includes dragons, magic, and the threat of the White Walkers.
  - **Political Intrigue:** Features alliances, betrayals, and complex political maneuvers.

- **Themes:**
  - **Power and Ambition:** Explores the lengths people go to gain and maintain power.
  - **Loyalty and Betrayal:** Highlights the importance and consequences of loyalty and bet

#### Invoke the agent with user_preferences

In [11]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)


agent = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

In [ ]:
for chunk in agent.stream(  
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in SF?"
            }
        ]
    },
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'id': 'call_YMdm7nmpQqt2izO0M3sBEbpd', 'name': 'get_weather', 'args': {'location': 'San Francisco, CA'}}]
step: tools
content: [{'type': 'text', 'text': 'Weather in San Francisco, CA: Sunny, 72°F'}]
step: model
content: [{'type': 'text', 'text': 'The weather in San Francisco, CA is currently sunny with a temperature of 72°F.'}]


In [13]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer  


def get_weather1(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()  
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"
 
agent = create_agent(
    model, 
    tools=[get_weather1],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom"
):
    print(chunk)

Looking up data for city: San Francisco
Acquired data for city: San Francisco


In [15]:
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"


agent = create_agent(
    model, 
    tools=[get_weather],
)

def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)
    # N.B. all content is available through token.content_blocks


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


input_message = {"role": "user", "content": "What is the weather in Boston?"}
for stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],  
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)  
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):  # `source` captures node name
                _render_completed_message(update["messages"][-1])

[{'name': 'get_weather', 'args': '', 'id': 'call_vvXWKrTmzYKMoXOvZryIFKaf', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'city', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '":"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'Boston', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_vvXWKrTmzYKMoXOvZryIFKaf', 'type': 'tool_call'}]
Tool response: [{'type': 'text', 'text': "It's always sunny in Boston!"}]
It's| always| sunny| in| Boston|!|

#### Advanced concepts

##### ToolStrategy

ToolStrategy uses artificial tool calling to generate structured output. This works with any model that supports tool calling:

In [131]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=model,
    tools=[search],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})


result["structured_response"]
# # ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

##### Memory

In [2]:
def demo_args(*args):
    print(args)  # args 是一个元组
    for i, value in enumerate(args, start=1):
        print(f"第{i}个参数: {value}")

import json 
demo_args(10, 20, 304)

(10, 20, 304)
第1个参数: 10
第2个参数: 20
第3个参数: 304


In [133]:
import json

def demo_kwargs(**kwargs):
    print(kwargs)  # kwargs 是一个字典
    for key1, value2 in kwargs.items():
        print(f"{key1} = {value2}")
   
    print(json.dumps(kwargs)) 
demo_kwargs(name="Alice", age=25, city="Shanghai")

{'name': 'Alice', 'age': 25, 'city': 'Shanghai'}
name = Alice
age = 25
city = Shanghai
{"name": "Alice", "age": 25, "city": "Shanghai"}


In [134]:
class MyClass:
    @staticmethod
    def static_method():
        print("This is a static method.")

    @classmethod
    def class_method(cls):
        print(f"This is a class method of {cls.__name__}.")

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value

# 使用
MyClass.static_method()
MyClass.class_method()

obj = MyClass()
obj.name = "Alice"
print(obj.name)

This is a static method.
This is a class method of MyClass.
Alice


In [15]:
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=800
)
from langchain_core.prompts.chat import HumanMessage

In [23]:
response = model.invoke([
  HumanMessage("What is machine learning?")
])
print(response.usage_metadata)

{'input_tokens': 12, 'output_tokens': 304, 'total_tokens': 316, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [26]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

||Par|rots| have| colorful| feathers| for| several| reasons|,| primarily| related| to| survival| and| reproduction|:

|1|.| **|M|ating| and| Attraction|**|:| Bright| and| colorful| feathers| are| often| used| to| attract| mates|.| In| many| par|rot| species|,| vibrant| plum|age| is| a| sign| of| health| and| genetic| fitness|,| making| individuals| with| such| traits| more| attractive| to| potential| mates|.

|2|.| **|Species| and| Individual| Recognition|**|:| The| distinct| color| patterns| help| parro|ts| recognize| members| of| their| own| species|,| which| is| important| for| social| interactions| and| breeding|.| It| can| also| help| individuals| recognize| each| other| within| fl|ocks|.

|3|.| **|Cam|ouflage|**|:| Although| it| might| seem| counter|int|uitive|,| the| bright| colors| can| actually| serve| as| camouflage| in| their| natural| habitats|.| Many| parro|ts| live| in| tropical| environments| where| the| play| of| light| through| the| canopy| and| the| colorful| surround

In [32]:
respsonse = model.stream("Why do parrots have colorful feathers?")
print(respsonse)

<generator object BaseChatModel.stream at 0x7f9b2d628b80>


## Streaming

### Basic text stream

In [28]:
prompt="How do airplanes stay in the air?"
    
for chunk in model.stream(prompt):
    print(chunk.content, end='')

Airplanes stay in the air primarily due to the principles of aerodynamics, specifically through the generation of lift. Lift is the force that directly opposes the weight of the airplane and holds it in the sky. Here's how it works:

1. **Wing Shape (Airfoil)**: The wings of an airplane are designed with a special shape known as an airfoil. An airfoil is curved on the top and flatter on the bottom. This shape helps in manipulating airflow to create lift.

2. **Bernoulli's Principle**: As the airplane moves forward, air flows over and under the wings. The air moving over the curved top of the wing travels faster than the air moving underneath. According to Bernoulli's Principle, faster-moving air results in lower pressure. Therefore, the pressure on top of the wing is lower than the pressure beneath it, creating lift.

3. **Newton's Third Law**: Lift is also explained by Newton's Third Law of Motion, which states that for every action, there is an equal and opposite reaction. As the win

As opposed to invoke(), which returns a single AIMessage after the model has finished generating its full response, stream() returns multiple AIMessageChunk objects, each containing a portion of the output text. Importantly, each chunk in a stream is designed to be gathered into a full message via summation:

In [26]:
full = None  # None | AIMessageChunk
for chunk in model.stream(prompt):
    full = chunk if full is None else full + chunk
    print(full.text)




Air
Airplanes
Airplanes stay
Airplanes stay in
Airplanes stay in the
Airplanes stay in the air
Airplanes stay in the air primarily
Airplanes stay in the air primarily due
Airplanes stay in the air primarily due to
Airplanes stay in the air primarily due to the
Airplanes stay in the air primarily due to the principles
Airplanes stay in the air primarily due to the principles of
Airplanes stay in the air primarily due to the principles of aer
Airplanes stay in the air primarily due to the principles of aerodynamics
Airplanes stay in the air primarily due to the principles of aerodynamics,
Airplanes stay in the air primarily due to the principles of aerodynamics, specifically
Airplanes stay in the air primarily due to the principles of aerodynamics, specifically through
Airplanes stay in the air primarily due to the principles of aerodynamics, specifically through the
Airplanes stay in the air primarily due to the principles of aerodynamics, specifically through the generation
Airplanes

In [30]:
async for event in model.astream_events("Hello"):

    if event["event"] == "on_chat_model_start":
        print(f"Input: {event['data']['input']}")

    elif event["event"] == "on_chat_model_stream":
        print(f"Token: {event['data']['chunk'].text}")

    elif event["event"] == "on_chat_model_end":
        print(f"Full message: {event['data']['output'].text}")

    else:
        pass

Input: Hello


Token: 
Token: 
Token: Hello
Token: !
Token:  How
Token:  can
Token:  I
Token:  assist
Token:  you
Token:  today
Token: ?
Token: 
Token: 
Full message: Hello! How can I assist you today?


In [27]:
print(full.content_blocks)

[{'type': 'text', 'text': "Airplanes stay in the air primarily due to the principles of aerodynamics, specifically through the generation of lift. Lift is the force that directly opposes the weight of the airplane and holds it in the sky. Here's how it works:\n\n1. **Airfoil Shape**: The wings of an airplane are designed with a special shape known as an airfoil. This shape is curved on the top and flatter on the bottom. As the airplane moves forward, air flows over and under the wings.\n\n2. **Bernoulli's Principle**: According to Bernoulli's principle, the pressure of a fluid decreases as its velocity increases. The air moving over the curved top of the wing travels faster than the air moving underneath. This creates lower pressure on top of the wing and higher pressure underneath, generating lift.\n\n3. **Newton's Third Law**: Newton's third law of motion states that for every action, there is an equal and opposite reaction. As the wings push air downwards, the air pushes the wings u

### Agent

#### Create Agent

In [14]:

from langchain.tools import tool

@tool
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]


from langchain.agents import create_agent

agent_weather = create_agent(
    model,
    tools = [get_weather],
    system_prompt=(
        "You are a helpful assistant help user to get the real time weather of a given city."
     
        "Use tool get_weather to fetch the current weather information."
    )
)

#### Agent progress

In [46]:
## stream_mode="updates"
for chunk in agent_weather.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in Shanghai?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'id': 'call_pd51SeOmV8PYTcDbI3QvyCQL', 'name': 'get_weather', 'args': {'latitude': 31.2304, 'longitude': 121.4737, 'city': 'Shanghai'}}]
step: tools
content: [{'type': 'text', 'text': '{"time": "2026-01-07T08:15", "interval": 900, "temperature_2m": 8.8, "wind_speed_10m": 9.5}'}]
step: model
content: [{'type': 'text', 'text': 'The current weather in Shanghai is 8.8°C with a wind speed of 9.5 km/h.'}]


#### LLM tokens 

In [33]:
## stream_mode="messages"
for token, metadata in agent_weather.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in Shanghai?"}]},
    stream_mode="messages",
):
    print(f"node: {metadata['langgraph_node']}")
    print(f"content: {token.content_blocks}")
    print("\n")

node: model
content: []


node: model
content: [{'type': 'tool_call_chunk', 'id': 'call_l4Yl8qDcDAW4FH5EYBSbjbz0', 'name': 'get_weather', 'args': '', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '{"', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'latitude', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '":', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '31', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '.', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '230', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '4', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ',"', 'inde

#### Custom updates

Use get_stream_writer to stream updates from tools as they are executed.

In [3]:
from langchain.tools import tool
from langgraph.config import get_stream_writer  

@tool
def get_weather(latitude, longitude, city):
    """This is a publically available API that returns the weather for a given location."""
    writer = get_stream_writer()  
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]


from langchain.agents import create_agent

agent_weather = create_agent(
    model,
    tools = [get_weather],
    system_prompt=(
        "You are a helpful assistant help user to get the real time weather of a given city."
     
        "Use tool get_weather to fetch the current weather information."
    )
)

In [5]:
## stream_mode="custom"
for chunk in agent_weather.stream(
    {"messages": [{"role": "user", "content": "What is the weather in Shanghai?"}]},
    stream_mode="custom"
):
    print(chunk)

Looking up data for city: Shanghai
Acquired data for city: Shanghai


#### Stream multiple modes

In [6]:
## stream_mode==["updates", "custom"]
for stream_mode, chunk in agent_weather.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in Shanghai?"}]},
    stream_mode=["updates", "custom"]
):
    print(f"stream_mode: {stream_mode}")
    print(f"content: {chunk}")
    print("\n")

stream_mode: updates
content: {'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_pr0VL3UcgpqXjAsP5MNW1eDC', 'function': {'arguments': '{"latitude":31.2304,"longitude":121.4737,"city":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 96, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CvchOjQxc2utDiSQYCpG66B8FSYX1', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b9c0a-762b-7962-b61b-b6bd8676a5ae-0', tool_calls=[{'name': 'get_weather', 'args': {'latitude': 31.2304, 'longitude': 121.4737, 'city': 'Shanghai'}, 'id': 'call_pr0VL3UcgpqXjAsP5MNW

In [12]:
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage


def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)
    # N.B. all content is available through token.content_blocks


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


input_message = {"role": "user", "content": "What is the weather in Boston?"}
for stream_mode, data in agent_weather.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],  
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)  
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):  # `source` captures node name
                _render_completed_message(update["messages"][-1])

[{'name': 'get_weather', 'args': '', 'id': 'call_WoNnJP67WQ0hZq4nmjbM2Zdu', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'latitude', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '":', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '42', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '.', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '360', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '1', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ',"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'longitude', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '":', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '-', 'id': None, 'index': 0,

## Chat models

### Amazon Claude

In [22]:
from langchain_core.messages import HumanMessage, AIMessage
from gen_ai_hub.proxy.langchain.amazon import ChatBedrock

async def async_amazon_chat_model():
    # Initialize the ChatBedrock model with the desired configuration
    chat_model = ChatBedrock(
        model_name="anthropic--claude-3.5-sonnet",
        model_kwargs={"temperature": 0.0}
    )
    # Send a message to the model
    response = await chat_model.ainvoke(
        [HumanMessage(content="Write me a song about sparkling water.")]
    )
    # Validate and print the response
    if isinstance(response, AIMessage):
        print("Response:", response.content)

await async_amazon_chat_model()

/tmp/ipykernel_30808/3555923443.py:6: UserWarning: WARNING! client_params is not default parameter.
                client_params was transferred to model_kwargs.
                Please confirm that client_params is what you intended.
  chat_model = ChatBedrock(


Response: Here's a fun, light-hearted song about sparkling water:

Verse 1:
Tiny bubbles rising up,
In my clear and frosty cup,
No sugar, no calories, just pure delight,
Sparkling water, oh so bright.

Chorus:
Fizzy, fizzy, can't you see?
You're the perfect drink for me,
Refreshing and crisp, with a subtle zing,
Sparkling water, makes my taste buds sing!

Verse 2:
Lemon, lime, or just plain,
You're my go-to to hydrate my brain,
A healthier choice, that's easy to see,
Sparkling water sets my spirit free.

(Repeat Chorus)

Bridge:
Some like it flavored, some like it plain,
But we all agree, it's not a strain,
On our waistlines or our teeth,
Sparkling water's such a relief!

(Repeat Chorus)

Outro:
Fizzy, fizzy, bubbling bright,
Sparkling water, pure delight!


### OpenAI GPT

In [6]:
from gen_ai_hub.proxy.langchain import ChatOpenAI
model = ChatOpenAI(  proxy_model_name='gpt-4o' ,
                    temperature=0.0,
                    verbose = False)
model.invoke('What is your name?')

AIMessage(content='I am ChatGPT, an AI language model created by OpenAI. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 12, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CxRHpazP5hDCGjtSt4eoq2pcoKKOl', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bb5d6-44a0-7dc1-9791-8323052ea82b-0', usage_metadata={'input_tokens': 12, 'output_tokens': 22, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from langchain_core.prompts.chat import (
    AIMessagePromptTemplate,
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')
chat_llm = ChatOpenAI(
    proxy_model_name='gpt-4o',
    proxy_client=proxy_client
)

template = 'You are a helpful assistant that translates English into Chinese.'

system_message_prompt = SystemMessagePromptTemplate.from_template(template)
example_human = HumanMessagePromptTemplate.from_template('Hi')
example_ai = AIMessagePromptTemplate.from_template('Ahoy!')
human_template = '{text}'
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

chat_prompt = ChatPromptTemplate.from_messages(
    [
        system_message_prompt,
        example_human, 
        example_ai,
        human_message_prompt
    ]
)

chain = chat_prompt | chat_llm

response = chain.invoke(
    {
        'text': 
        '''
         Bosch secured a massive, multi-billion dollar deal with Toyota for Advanced Driver Assistance Systems (ADAS) to power next-gen safety features, leveraging Qualcomm chips for a system hitting 2026 Euro NCAP 5-star ratings, with mass production starting in 2028 for global markets like North America, Europe, and Japan. 

        This significant partnership focuses on scalable, chip-based solutions for high-level driver assistance, aiming for advanced safety and comfort in Toyota's future vehicles, marking a major win for Bosch in the competitive intelligent driving sector.

        Bosch's China Momentum: The deal follows Bosch's aggressive expansion in China, where it has already secured more than half a dozen ADAS customers, including BAIC, Dongfeng, and Jetour, and recently achieved production milestones for high-level ADAS solutions in partnership with WeRide.
        '''      
    }
)

print(response.content)


博世与丰田签署了一项巨额的多亿美元合同，为高级驾驶辅助系统（ADAS）提供支持，以推动下一代安全功能，这些系统将利用高通的芯片，目标是在2026年达到欧洲NCAP五星级评级，并计划在2028年开始生产，以供应北美、欧洲和日本等全球市场。

这一重要合作伙伴关系专注于可扩展的芯片解决方案，用于高水平的驾驶辅助，旨在提升丰田未来车辆的安全性和舒适性，这标志着博世在竞争激烈的智能驾驶领域取得了重大胜利。

博世在中国的势头：这一交易紧随博世在中国的积极扩张，在该市场中，博世已获得包括北汽、东风和捷途在内的多位ADAS客户，并且最近与文远知行合作的高级ADAS解决方案生产取得了里程碑式的成就。


## Short-term memory

### Example: InMemorySaver

#### Create a new agent with checkpointer

In [ ]:

from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver  

@tool
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]


from langchain.agents import create_agent

agent_weather2 = create_agent(
    model,
    tools = [get_weather],
    system_prompt=(
        "You are a helpful assistant help user to get the real time weather of a given city."
        "Use tool get_weather to fetch the current weather information."
    ),
    
    #Specify a checkpointer when creating an agent.
    checkpointer=InMemorySaver(),
)

#### Test

##### Test the agent created previously  without the checkpointer 

In [60]:
response= agent_weather.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the capital city of China?"}
        ]
    },
)
print(response['messages'][-1].content)

The capital city of China is Beijing.


In [61]:
response= agent_weather.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather of this city now?"}
        ]
    },
)
print(response['messages'][-1].content)

Please provide the name of the city or its latitude and longitude coordinates so I can fetch the current weather information for you.


##### Test the new agent created with the checkpointer 

Beware the different thread_id value

In [53]:
response= agent_weather2.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the capital city of China?"}
        ]
    },
    {
        "configurable": {"thread_id": "1"}
    },  
)
print(response['messages'][-1].content)


The capital city of China is Beijing.


In [54]:
response= agent_weather2.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the capital city of Germany?"}
        ]
    },
    {
        "configurable": {"thread_id": "2"}
    },  
)
print(response['messages'][-1].content)


The capital city of Germany is Berlin.


In [59]:
response= agent_weather2.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather of this city now?"}
        ]
    },
    {
        "configurable": {"thread_id": "1"}
    },  
)
print(response['messages'][-1].content)


The current weather in Beijing is as follows:
- Temperature: 2.3°C
- Wind Speed: 9.8 m/s


In [62]:
response= agent_weather2.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather of this city now?"}
        ]
    },
    {
        "configurable": {"thread_id": "2"}
    },  
)
print(response['messages'][-1].content)


The current weather in Berlin is as follows:
- Temperature: 0.4°C
- Wind Speed: 4.1 m/s


### Access memory

#### Customizing agent memory

Use <i>AgentState</i> to manage short term memory, specifically the conversation history via a messages key. <br>

In [208]:
class CustomAgentState(AgentState):  
    user_id: str
    add_info: dict

#### Read short-term memory in a tool

Custom state schemas are passed to create_agent using the <i>state_schema</i> parameter.<br>
Access short term memory (state) in a tool using the <i>runtime</i> parameter (typed as ToolRuntime). <br>
The runtime parameter is hidden from the tool signature (so the model doesn’t see it), but the tool can access the state through it.

In [231]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user info."""
    user_id = runtime.state["user_id"]
    if user_id == "user_123":
        user="John Smith"
    elif user_id == "user_124":
        user="Henry Thomas"
    else:
        user="N/A"
    return f"User is{user}"

agent = create_agent(
    model,
    tools=[get_user_info],
    state_schema=CustomAgentState,  
    checkpointer=InMemorySaver(),
)

In [235]:
# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello"}],
        "user_id": "user_123",  
        "add_info": {"email": "John.Smith@gmail.com"}  
    },
    {"configurable": {"thread_id": "1"}})

print(result["messages"][-1].content)

Hello! How can I help you today?


In [236]:
# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello, what is the email address of the user?"}],
        #"user_id": "user_124", 
        "add_info": {"email": "Henry.Thomas@outlook.com"}  
     
    },
    {"configurable": {"thread_id": "1"}})

print(result["messages"][-1].content)

The user's name is John Smith, but I don't have access to their email address. If you need further assistance, feel free to ask!


"add_info": {"email": "John.Smith@gmail.com"}  
"add_info": {"email": "Henry.Thomas@outlook.com"}  
"user_id": "user_124", 

### Example: InMemorySaver

In [ ]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):  
    user_id: str
    current_month: str
    current_city: str
    preferences: dict



agent_weather3 = create_agent(
    model,
    tools = [get_weather],
    system_prompt=(
        "You are a helpful assistant help user to get the real time weather of a given city."
        "Use tool get_weather to fetch the current weather information."
    ),
    
    state_schema=CustomAgentState, 
    
    #Specify a checkpointer when creating an agent.
    checkpointer=InMemorySaver(),
)


In [142]:
# Custom state can be passed in invoke
response = agent_weather3.invoke(
    {
        "messages": [{"role": "user", "content": "Hello,what is the current city now?"}],
        "user_id": "user_123",  
        "current_month":"July",
        "current_city":"Sydney",
        "preferences": {"theme": "dark"}  
    },
    {"configurable": {"thread_id": "1"}})

print(response['messages'][-1].content)

I don't have access to your current location. You can provide me with the name of the city or its latitude and longitude, and I can help you find the current weather for that location.


In [ ]:
response= agent_weather3.invoke(
    {
        "messages": [
            {"role": "user", "content": "what is the weather of the city?"}
        ]
    },
    {
        "configurable": {"thread_id": "1"}
    },  
)
print(response['messages'][-1].content)


It is October 2023.


In [137]:
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime


class CustomState(AgentState):
    user_id: str
    preferences: dict

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user info."""
    user_id = runtime.state["user_id"]
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

agent = create_agent(
    model,
    tools=[get_user_info],
    state_schema=CustomState,
)

result = agent.invoke({
    "messages": "look up user information",
    "user_id": "user_123"
})
print(result["messages"][-1].content)
# > User is John Smith.

The user information is: John Smith.


### Access memory

In [143]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):  
    user_id: str
    preferences: dict

agent = create_agent(
    model,
    tools=[get_user_info],
    state_schema=CustomAgentState,  
    checkpointer=InMemorySaver(),
)

# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello, what is the prefered theme of the user?"}],
        "user_id": "user_123",  
        "preferences": {"theme": "dark"}  
    },
    {"configurable": {"thread_id": "1"}})

print(result["messages"][-1].content)

The preferred theme of the user, John Smith, is not specified in the available information.
